In [1]:
from datetime import datetime, timedelta, timezone
from pyspark.sql import functions as F
from pyspark.sql.types import IntegerType, LongType, StringType

spark.conf.set("spark.sql.session.timeZone", "UTC")

# Query URI: KQL database > Copy query URI (form https://<...>.kusto.data.microsoft.com)
KUSTO_URI = "https://trd-s733h2caqmq2f44yrb.z3.kusto.fabric.microsoft.com"
KUSTO_DB = "eh_fleet"
SIGNALS = ["gps", "engine", "fuel", "temperature", "device_status"]
SAFETY_LAG_MIN = 2
WM_TABLE = "ops.ops_bronze_load_watermark"

StatementMeta(, ff22f24e-89a0-4a3a-a785-1fb2dec8c8c8, 3, Finished, Available, Finished, False)

In [2]:
spark.sql("CREATE SCHEMA IF NOT EXISTS ops")
spark.sql(f"""CREATE TABLE IF NOT EXISTS {WM_TABLE}
              (signal STRING, watermark_utc STRING, updated_at TIMESTAMP) USING DELTA""")

FMT = "%Y-%m-%dT%H:%M:%SZ"

def get_wm(signal):
    row = spark.sql(f"SELECT watermark_utc FROM {WM_TABLE} WHERE signal='{signal}'").first()
    s = row[0] if row else "1900-01-01T00:00:00Z"
    return datetime.strptime(s, FMT).replace(tzinfo=timezone.utc)

def set_wm(signal, wm):
    spark.sql(f"DELETE FROM {WM_TABLE} WHERE signal='{signal}'")
    spark.sql(f"INSERT INTO {WM_TABLE} VALUES ('{signal}', '{wm.strftime(FMT)}', current_timestamp())")

def load_signal(signal, upper):
    kql_table = f"bronze_{signal}_raw"          # KQL table: no schema in KQL, name unchanged
    target = f"bronze.bronze_{signal}_raw"      # Lakehouse Delta table: schema.table
    lower = get_wm(signal)
    if lower >= upper:
        print(signal, "nothing to do"); return 0

    query = f"""
    {kql_table}
    | where ingestion_time() >= datetime({lower.strftime(FMT)})
        and ingestion_time() <  datetime({upper.strftime(FMT)})
    | project raw_value, ingest_ts = ingestion_time()
    """
    df = (spark.read.format("com.microsoft.kusto.spark.synapse.datasource")
          .option("accessToken", mssparkutils.credentials.getToken(KUSTO_URI))
          .option("kustoCluster", KUSTO_URI)
          .option("kustoDatabase", KUSTO_DB)
          .option("kustoQuery", query)
          .load())

    out = (df.withColumnRenamed("ingest_ts", "ingestion_time")
             .withColumn("source_stream", F.lit(signal))
             .withColumn("source_type", F.lit("STREAM"))
             .withColumn("ingestion_date", F.to_date("ingestion_time"))
             .withColumn("kafka_partition", F.lit(None).cast(IntegerType()))
             .withColumn("kafka_offset", F.lit(None).cast(LongType()))
             .withColumn("source_file", F.lit(None).cast(StringType()))
             .withColumn("file_line", F.lit(None).cast(IntegerType()))
             .select("raw_value", "source_stream", "source_type", "ingestion_time",
                     "ingestion_date", "kafka_partition", "kafka_offset",
                     "source_file", "file_line")
             .cache())
    n = out.count()

    # Retry guard: removes only rows of THIS window, in case an earlier attempt
    # wrote rows but failed before saving the watermark.
    if spark.catalog.tableExists(target):
        spark.sql(f"""DELETE FROM {target}
                      WHERE ingestion_time >= TIMESTAMP'{lower:%Y-%m-%d %H:%M:%S}'
                        AND ingestion_time <  TIMESTAMP'{upper:%Y-%m-%d %H:%M:%S}'""")

    out.write.format("delta").mode("append").partitionBy("ingestion_date").saveAsTable(target)
    set_wm(signal, upper)
    out.unpersist()
    print(signal, "loaded", n)
    return n

StatementMeta(, ff22f24e-89a0-4a3a-a785-1fb2dec8c8c8, 4, Finished, Available, Finished, False)

In [3]:
spark.table(WM_TABLE).show()      # check this before running, see the warning below
upper = (datetime.now(timezone.utc) - timedelta(minutes=SAFETY_LAG_MIN)).replace(microsecond=0)
for s in SIGNALS:
    load_signal(s, upper)
for s in SIGNALS:
    print(s, spark.table(f"bronze.bronze_{s}_raw").count())
mssparkutils.notebook.exit("done")

StatementMeta(, ff22f24e-89a0-4a3a-a785-1fb2dec8c8c8, 5, Finished, Available, Finished, False)

gps loaded 30895
engine loaded 30926
fuel loaded 5176
temperature loaded 7197
device_status loaded 0
gps 30895
engine 30926
fuel 5176
temperature 7197
device_status 0


StatementMeta(, ff22f24e-89a0-4a3a-a785-1fb2dec8c8c8, 6, Finished, Available, Finished, False)

In [ ]:
# Release cached data and clear Spark cache
spark.catalog.clearCache()